# TRM Compiler Pass Ordering — Real LLVM Training

Train a 60K-param TRM model on **real LLVM** via CompilerGym, benchmark vs LLVM -Oz/-O3.

**Before running:** Go to Runtime → Change runtime type → Select **GPU** (T4 or higher)

In [16]:
# Cell 0: Restart runtime (clears any broken venv from previous runs)
import os

# Delete old venv if it exists
import subprocess
if os.path.exists('/content/trm-env'):
    print("Removing old venv...")
    subprocess.run(['rm', '-rf', '/content/trm-env'], check=True)
    print("Old venv removed.")

print("Now go to Runtime → Restart runtime and run all cells fresh!")
print("IMPORTANT: Don't run this cell again after restart - just run cells 1-4.")

Removing old venv...
Old venv removed.
Now go to Runtime → Restart runtime and run all cells fresh!
IMPORTANT: Don't run this cell again after restart - just run cells 1-4.


In [1]:
# Cell 1: Clone repo
import os
import subprocess

PROJECT_DIR = '/content/trm-youtubevids'
VENV_DIR = '/content/trm-env'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Cree0618/trm-youtubevids.git', PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /content/trm-youtubevids


In [2]:
# Cell 2: Create Python 3.11 venv and install dependencies
# compiler_gym requires Python <= 3.11 (uses distutils removed in 3.12)

import subprocess
import os

# Create venv if needed
if not os.path.exists(VENV_DIR):
    result = subprocess.run(['python3.11', '-m', 'venv', VENV_DIR], capture_output=True, text=True)
    print(f"Created venv: {result.returncode == 0}")
else:
    print(f"Venv exists: {VENV_DIR}")

VENV_PIP = f'{VENV_DIR}/bin/pip'
VENV_PYTHON = f'{VENV_DIR}/bin/python'

# Upgrade pip
subprocess.run([VENV_PIP, 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

# Install torch (CPU version to avoid CUDA issues)
subprocess.run([VENV_PIP, 'install', 'torch', 'numpy', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)

# Install pydantic, protobuf, and other required deps
subprocess.run([VENV_PIP, 'install', 'pydantic', 'protobuf==3.20.3', 'requests', 'docker', 'fasteners', 'absl-py', 'deprecated', 'tabulate'], check=True)

# Install grpcio NEWER version (has pre-built wheels for Python 3.11)
# The trick: install newer grpcio FIRST, then install compiler_gym with --no-deps
subprocess.run([VENV_PIP, 'install', 'grpcio'], check=True)

# Install compiler_gym without dependencies
subprocess.run([VENV_PIP, 'install', 'compiler_gym', '--no-deps'], check=True)

# Verify
result = subprocess.run([VENV_PYTHON, '-c', 'import compiler_gym; print(f"CompilerGym: {compiler_gym.__version__}")'], capture_output=True, text=True)
print(result.stdout.strip())
if result.returncode != 0:
    print(f"Error: {result.stderr}")

Created venv: True

Error: Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/trm-env/lib/python3.11/site-packages/compiler_gym/__init__.py", line 32, in <module>
    from compiler_gym.compiler_env_state import (
  File "/content/trm-env/lib/python3.11/site-packages/compiler_gym/compiler_env_state.py", line 12, in <module>
    import requests
ModuleNotFoundError: No module named 'requests'



In [3]:
# Cell 3: Quick smoke test - verify CompilerGym works

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'

test_code = '''
import compiler_gym
env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort",
    observation_space="Autophase", reward_space="IrInstructionCountOz")
obs = env.reset()
ap = env.observation["Autophase"]
ic = env.observation["IrInstructionCount"]
print(f"Autophase: {len(ap)} features, Initial inst: {ic}")
for i in range(3):
    _, reward, done, _ = env.step(env.action_space.sample())
    inst = env.observation["IrInstructionCount"]
    print(f"  Step {i}: inst={inst} reward={reward:.2f} done={done}")
env.close()
print("CompilerGym OK!")
'''

result = subprocess.run([VENV_PYTHON, '-c', test_code], capture_output=False, text=True)

In [4]:
# Cell 4: Run TRM on REAL LLVM - train and benchmark

import subprocess

VENV_PYTHON = '/content/trm-env/bin/python'
PROJECT_DIR = '/content/trm-youtubevids'

cmd = [
    VENV_PYTHON,
    f'{PROJECT_DIR}/trm_compiler_real_llvm.py',
    '--epochs', '10',
    '--episodes', '10',
    '--benchmarks', 'qsort', 'adpcm',
    '--max-steps', '20',
    '--batch-size', '64',
    '--num-random', '20',
    '--seed', '42'
]

# Note: No --synthetic flag = uses real CompilerGym
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])


Done. Exit code: CompletedProcess(args=['/content/trm-env/bin/python', '/content/trm-youtubevids/trm_compiler_real_llvm.py', '--epochs', '10', '--episodes', '10', '--benchmarks', 'qsort', 'adpcm', '--max-steps', '20', '--batch-size', '64', '--num-random', '20', '--seed', '42'], returncode=0)


## Results

The script outputs benchmark tables comparing:
- **LLVM-Oz**: Fixed -Oz pipeline
- **LLVM-O3**: Fixed -O3 pipeline
- **Random(N)**: Best of N random trials
- **TRM-loop**: TRM with closed-loop feedback
- **TRM-blind**: TRM without feedback